In [6]:
import pandas as pd
import numpy as np

loading data

In [20]:
data1 = pd.read_csv("../data/202609-divvy-tripdata.csv")
data1.info()

data2 = pd.read_csv("../data/202309-divvy-tripdata.csv")
data2.info()
data = pd.concat(
    [
        data1.assign(source_file="202609-divvy-tripdata.csv"),
        data2.assign(source_file="202309-divvy-tripdata.csv")
    ],
    ignore_index=True
)

<class 'pandas.DataFrame'>
RangeIndex: 770562 entries, 0 to 770561
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   ride_id             770562 non-null  str    
 1   rideable_type       770562 non-null  str    
 2   started_at          770562 non-null  str    
 3   ended_at            770562 non-null  str    
 4   start_station_name  610221 non-null  str    
 5   start_station_id    610221 non-null  str    
 6   end_station_name    600529 non-null  str    
 7   end_station_id      600529 non-null  str    
 8   start_lat           770562 non-null  float64
 9   start_lng           770562 non-null  float64
 10  end_lat             769918 non-null  float64
 11  end_lng             769918 non-null  float64
 12  member_casual       770562 non-null  str    
dtypes: float64(4), str(9)
memory usage: 76.4 MB
<class 'pandas.DataFrame'>
RangeIndex: 666371 entries, 0 to 666370
Data columns (total 13 columns):
 

Per-File Categorical & ID Format Comparison

In [21]:
cat_cols = ["rideable_type", "member_casual"]
for col in cat_cols:
    print(f"\n--- {col} distribution by source_file ---")
    print(pd.crosstab(data["source_file"], data[col], dropna=False))

# 2. Compare station ID formats (string length & numeric vs alphanumeric patterns)
id_cols = ["start_station_id", "end_station_id"]
for col in id_cols:
    print(f"\n--- {col} format summary by source_file ---")
    temp = data[["source_file", col]].dropna().copy()
    temp[col] = temp[col].astype(str)
    temp["len"] = temp[col].str.len()
    temp["is_digits_only"] = temp[col].str.isdigit()

    summary = temp.groupby("source_file").agg(
        non_null_count=(col, "count"),
        unique_ids=(col, "nunique"),
        min_len=("len", "min"),
        max_len=("len", "max"),
        pct_digits_only=("is_digits_only", lambda s: round(s.mean() * 100, 2)),
        sample_values=(col, lambda s: list(s.unique()[:5]))
    )
    print(summary)


--- rideable_type distribution by source_file ---
rideable_type              classic_bike  electric_bike
source_file                                           
202309-divvy-tripdata.csv        347372         318999
202609-divvy-tripdata.csv        189924         580638

--- member_casual distribution by source_file ---
member_casual              casual  member
source_file                              
202309-divvy-tripdata.csv  261635  404736
202609-divvy-tripdata.csv  294120  476442

--- start_station_id format summary by source_file ---
                           non_null_count  unique_ids  min_len  max_len  \
source_file                                                               
202309-divvy-tripdata.csv          565059        1241        3       13   
202609-divvy-tripdata.csv          610221        1554        8        8   

                           pct_digits_only  \
source_file                                  
202309-divvy-tripdata.csv            45.17   
202609-divvy-tr

The ride_id Investigation

In [22]:
total_rows = len(data)
unique_rides = data["ride_id"].nunique(dropna=False)
print(f"Total rows: {total_rows:,} | Unique ride_ids: {unique_rides:,} | Duplicates: {total_rows - unique_rides:,}")

# Inspect duplicates if any exist
if total_rows != unique_rides:
    dups = data[data.duplicated(subset=["ride_id"], keep=False)].sort_values("ride_id")
    print("\nDuplicate ride_id records:")
    print(dups[["source_file", "ride_id", "started_at", "ended_at"]].head(10))

# Check string length and hex pattern per file
ride_id_check = data.assign(
    id_len=data["ride_id"].astype(str).str.len(),
    is_hex=data["ride_id"].astype(str).str.match(r"^[0-9A-Fa-f]+$")
)

print("\n--- ride_id format by source_file ---")
print(ride_id_check.groupby("source_file").agg(
    null_count=("ride_id", lambda s: s.isna().sum()),
    lengths=("id_len", lambda s: sorted(s.unique().tolist())),
    pct_hex=("is_hex", lambda s: round(s.mean() * 100, 2))
))

Total rows: 1,436,933 | Unique ride_ids: 1,436,933 | Duplicates: 0

--- ride_id format by source_file ---
                           null_count lengths  pct_hex
source_file                                           
202309-divvy-tripdata.csv           0    [16]    100.0
202609-divvy-tripdata.csv           0    [16]    100.0


3. Coordinate Extremes & Rounding Investigation

In [23]:
coord_cols = ["start_lat", "start_lng", "end_lat", "end_lng"]

# 1. Min, Max, and Nulls per file (flagging 0.0 values)
print("--- Coordinate Min/Max & Zeros by source_file ---")
for col in coord_cols:
    stats = data.groupby("source_file")[col].agg(
        nulls=lambda s: s.isna().sum(),
        zeros=lambda s: (s == 0).sum(),
        min_val="min",
        max_val="max"
    )
    print(f"\nColumn: {col}")
    print(stats)

# 2. Decimal precision / rounding check
def count_decimals(series):
    return (
        series.dropna()
        .astype(str)
        .str.split(".")
        .str[1]
        .fillna("")
        .str.len()
    )

print("\n--- Coordinate Decimal Precision Distribution ---")
for col in coord_cols:
    dec_lengths = count_decimals(data[col])
    summary = pd.crosstab(data["source_file"], dec_lengths, rownames=["source_file"], colnames=["decimals"])
    print(f"\nDecimal places for {col}:")
    print(summary)


--- Coordinate Min/Max & Zeros by source_file ---

Column: start_lat
                           nulls  zeros    min_val  max_val
source_file                                                
202309-divvy-tripdata.csv      0      0  41.648501    42.07
202609-divvy-tripdata.csv      0      0  41.648501    42.07

Column: start_lng
                           nulls  zeros  min_val    max_val
source_file                                                
202309-divvy-tripdata.csv      0      0   -87.94 -87.528232
202609-divvy-tripdata.csv      0      0   -87.87 -87.528232

Column: end_lat
                           nulls  zeros  min_val  max_val
source_file                                              
202309-divvy-tripdata.csv    838      0    41.62    42.12
202609-divvy-tripdata.csv    644      0    41.61    42.18

Column: end_lng
                           nulls  zeros  min_val    max_val
source_file                                                
202309-divvy-tripdata.csv    838      0   -87.